# Bike Rental Demand Forecasting

This project forecasts hourly bike rental demand using statistical time-series models, classical machine learning, and deep learning.

## Dataset Features

| Feature      | Description                                |
| ------------ | ------------------------------------------ |
| `instant`    | Record index                               |
| `dteday`     | Date                                       |
| `season`     | Season category                            |
| `yr`         | Year                                       |
| `mnth`       | Month                                      |
| `hr`         | Hour of the day                            |
| `holiday`    | Whether the day is a holiday               |
| `weekday`    | Day of the week                            |
| `workingday` | Whether the day is a working day           |
| `weathersit` | Weather condition                          |
| `temp`       | Normalized temperature                     |
| `atemp`      | Normalized feeling temperature             |
| `hum`        | Normalized humidity                        |
| `windspeed`  | Normalized wind speed                      |
| `casual`     | Casual user rentals                        |
| `registered` | Registered user rentals                    |
| `cnt`        | **Total bike rentals — prediction target** |

### Project Workflow

**Preprocessing → EDA → Statistical Analysis → Classical Time Series → ML → Deep Learning → Model Comparison → Streamlit App**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
DATA_PATH = DATA_DIR / "hour.csv"

print(f"Dataset path: {DATA_PATH}")

Dataset path: ../data/hour.csv


In [5]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (17379, 17)


In [6]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [7]:
df.tail()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
17374,17375,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,11,108,119
17375,17376,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,8,81,89
17376,17377,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,7,83,90
17377,17378,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,13,48,61
17378,17379,2012-12-31,1,1,12,23,0,1,1,1,0.26,0.2727,0.65,0.1343,12,37,49


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.4 MB


### nothing is missing

In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
instant,17379.0,8690.000000,5017.029500,1.00,4345.5000,8690.0000,13034.5000,17379.0000
season,17379.0,2.501640,1.106918,1.00,2.0000,3.0000,3.0000,4.0000
yr,17379.0,0.502561,0.500008,0.00,0.0000,1.0000,1.0000,1.0000
mnth,17379.0,6.537775,3.438776,1.00,4.0000,7.0000,10.0000,12.0000
hr,17379.0,11.546752,6.914405,0.00,6.0000,12.0000,18.0000,23.0000
holiday,17379.0,0.028770,0.167165,0.00,0.0000,0.0000,0.0000,1.0000
weekday,17379.0,3.003683,2.005771,0.00,1.0000,3.0000,5.0000,6.0000
workingday,17379.0,0.682721,0.465431,0.00,0.0000,1.0000,1.0000,1.0000
weathersit,17379.0,1.425283,0.639357,1.00,1.0000,1.0000,2.0000,4.0000
temp,17379.0,0.496987,0.192556,0.02,0.3400,0.5000,0.6600,1.0000


### only registred & cnt has gap between mean and median , possble skewness

In [13]:
# missing vlaues test
missing = df.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [11]:
missing_percentage = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_percentage[missing_percentage > 0]

Series([], dtype: float64)

In [12]:
# duplication test
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [14]:
# unique values test
categorical_columns = [
    "season",
    "yr",
    "mnth",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(sorted(df[column].unique()))


season:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

yr:
[np.int64(0), np.int64(1)]

mnth:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

holiday:
[np.int64(0), np.int64(1)]

weekday:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

workingday:
[np.int64(0), np.int64(1)]

weathersit:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [16]:
# add proper datetime stamp
df["datetime"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(
    df["hr"], 
    unit="h"
)

df[["dteday", "hr", "datetime"]].head(10)

,dteday,hr,datetime
0,2011-01-01,0,2011-01-01 00:00:00
1,2011-01-01,1,2011-01-01 01:00:00
2,2011-01-01,2,2011-01-01 02:00:00
3,2011-01-01,3,2011-01-01 03:00:00
4,2011-01-01,4,2011-01-01 04:00:00
5,2011-01-01,5,2011-01-01 05:00:00
6,2011-01-01,6,2011-01-01 06:00:00
7,2011-01-01,7,2011-01-01 07:00:00
8,2011-01-01,8,2011-01-01 08:00:00
9,2011-01-01,9,2011-01-01 09:00:00


In [17]:
# sort crnologically by datetime
df = df.sort_values("datetime").reset_index(drop=True)

print(f"Chronological order: {df['datetime'].is_monotonic_increasing}")

Chronological order: True


In [19]:
df.head(5)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,datetime
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00


In [20]:
# set datetime as index
df = df.set_index("datetime")

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
datetime,,,,,,,,,,,,,,,,,
2011-01-01 00:00:00,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2011-01-01 01:00:00,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2011-01-01 02:00:00,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
2011-01-01 03:00:00,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
2011-01-01 04:00:00,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [21]:
# find start and end date of the dataset
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")
print(f"Duration: {df.index.max() - df.index.min()}")

Start: 2011-01-01 00:00:00
End:   2012-12-31 23:00:00
Duration: 730 days 23:00:00


In [22]:
# verify total rental demand equals casual + registered
target_check = (df["casual"] + df["registered"]) == df["cnt"]

print(f"Rows satisfying casual + registered = cnt: {target_check.mean():.2%}")

Rows satisfying casual + registered = cnt: 100.00%


In [23]:
# idenify gaps in hourly data
time_differences = df.index.to_series().diff()

time_differences.value_counts().head(10)


datetime
0 days 01:00:00    17303
0 days 02:00:00       64
0 days 03:00:00        6
0 days 13:00:00        1
0 days 23:00:00        1
0 days 07:00:00        1
0 days 14:00:00        1
1 days 13:00:00        1
Name: count, dtype: int64

In [24]:
gaps = time_differences[time_differences > pd.Timedelta(hours=1)]

print(f"Number of gaps greater than one hour: {len(gaps)}")

Number of gaps greater than one hour: 75


In [25]:
# basic target varable analysis
print(f"Minimum demand: {df['cnt'].min()}")
print(f"Maximum demand: {df['cnt'].max()}")
print(f"Mean demand:    {df['cnt'].mean():.2f}")
print(f"Median demand:  {df['cnt'].median():.2f}")

Minimum demand: 1
Maximum demand: 977
Mean demand:    189.46
Median demand:  142.00


### target is right shewed

In [26]:
# save data
OUTPUT_PATH = DATA_DIR / "hour_cleaned.csv"

df.to_csv(OUTPUT_PATH)

print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

Saved cleaned dataset to: ../data/hour_cleaned.csv
